## Solving a complex task with a multi-agent hierarchy

Find all Batman filming locations in the world, calculate the time to transfer via a cargo plane to there, and represent them on a map, with a color varying by a cargo plane transfer time. Also represent some supercar factories with the same cargo plane transfer time.

In [1]:
%pip install "smolagents[litellm]" plotly geopandas shapely kaleido -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
# We first make a tool to get the cargo plane transfer time.
import math
from typing import Optional, Tuple

from smolagents import tool


@tool
def calculate_cargo_travel_time(
    origin_coords: Tuple[float, float],
    destination_coords: Tuple[float, float],
    cruising_speed_kmh: Optional[float] = 750.0,  # Average speed for cargo planes
) -> float:
    """
    Calculate the travel time for a cargo plane between two points on Earth using great-circle distance.

    Args:
        origin_coords: Tuple of (latitude, longitude) for the starting point
        destination_coords: Tuple of (latitude, longitude) for the destination
        cruising_speed_kmh: Optional cruising speed in km/h (defaults to 750 km/h for typical cargo planes)

    Returns:
        float: The estimated travel time in hours

    Example:
        >>> # Chicago (41.8781° N, 87.6298° W) to Sydney (33.8688° S, 151.2093° E)
        >>> result = calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093))
    """

    def to_radians(degrees: float) -> float:
        return degrees * (math.pi / 180)

    # Extract coordinates
    lat1, lon1 = map(to_radians, origin_coords)
    lat2, lon2 = map(to_radians, destination_coords)

    # Earth's radius in kilometers
    EARTH_RADIUS_KM = 6371.0

    # Calculate great-circle distance using the haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    distance = EARTH_RADIUS_KM * c

    # Add 10% to account for non-direct routes and air traffic controls
    actual_distance = distance * 1.1

    # Calculate flight time
    # Add 1 hour for takeoff and landing procedures
    flight_time = (actual_distance / cruising_speed_kmh) + 1.0

    # Format the results
    return round(flight_time, 2)


print(calculate_cargo_travel_time((41.8781, -87.6298), (-33.8688, 151.2093)))

C:\Users\sobha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


22.82


In [4]:
import os
from dotenv import load_dotenv
from PIL import Image
from smolagents import CodeAgent, GoogleSearchTool, InferenceClientModel, VisitWebpageTool, WebSearchTool

load_dotenv()  # reads HF_TOKEN from the .env file next to this notebook
hf_token = os.environ.get("HF_TOKEN")

model = InferenceClientModel(model_id="Qwen/Qwen3-Coder-30B-A3B-Instruct", token=hf_token)

In [5]:
task = """Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.
Also give me some supercar factories with the same cargo plane transfer time."""

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads HF_TOKEN from the .env file next to this notebook

hf_token = os.environ.get("HF_TOKEN")

agent = CodeAgent(
    tools=[WebSearchTool(engine="bing"), VisitWebpageTool(), calculate_cargo_travel_time], 
    model=InferenceClientModel(
        # model_id="openai/gpt-oss-120b", 
        model_id="Qwen/Qwen3-Coder-30B-A3B-Instruct", 
        token=hf_token,
    ),
    additional_authorized_imports=["pandas"],
    max_steps=20,
)

agent.run(task)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Coder-30B-A3B-Instruct ──────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import pandas as pd                                                                                              
  from math import radians, cos, sin, asin, sqrt                                                                   
                                                                                                                   
  # Gotham coordinates                                                                                             
  gotham_coords = (40.7128, -74.0060)                                                                              
                                                                                                                   
  # Search for Batman filming locations                                                                            
  batman_locations = web_search("Batman filming locations worldwide")                                              
                                                                                                                   
  print("Batman filming locations:")                                                                               
  print(batman_locations[:500])  # First 500 characters to see the results                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Batman filming locations:
## Search Results

[Batman – Wikipedia](https://fi.wikipedia.org/wiki/Batman)
Batman eli Lepakkomies on yhdysvaltalaisen DC Comics - sarjakuvakustantamon julkaisemissa sarjakuvissa esiintyvä 
supersankari. Piirtäjä Bob Kanen ja käsikirjoittaja Bill Fingerin luoma Batman esiintyi ensimmäisen kerran vuoden 
1939 toukokuussa Detective Comics -sarjakuvalehden numerossa 27.

[Batman - Wikipedia](https://en.wikipedia.org/wiki/Batman)
Batman received a spin-off publication, Batman, in 1940. Kane and Fing

Out: None

[Step 1: Duration 12.88 seconds| Input tokens: 2,315 | Output tokens: 173]

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Final results - Batman filming locations                                                                       
  print("BATMAN FILMING LOCATIONS WITH CARGO PLANE TRANSFER TIMES FROM GOTHAM:")                                   
  print("=" * 60)                                                                                                  
  for index, row in df_batman.iterrows():                                                                          
      print(f"{row['Location']:>25} - {row['Travel_Time_Hours']:>8.2f} hours")                                     
                                                                                                                   
  print("\nSUPERCAR FACTORIES WITH SIMILAR CARGO PLANE TRANSFER TIMES:")                                           
  print("=" * 60)                                                                                                  
  # Find supercar factories with travel times similar to Batman locations                                          
  # Let's identify some interesting matches based on travel times                                                  
  print("Supercar factories grouped by travel time ranges:")                                                       
                                                                                                                   
  # Group by approximate travel time ranges                                                                        
  for time_range in [0, 5, 10, 15, 20]:                                                                            
      print(f"\nApproximately {time_range} to {time_range + 5} hours:")                                            
      for index, row in df_supercars.iterrows():                                                                   
          if time_range <= row['Travel_Time_Hours'] < time_range + 5:                                              
              print(f"  - {row['Factory']} ({row['Travel_Time_Hours']:.2f} hours)")                                
                                                                                                                   
  # Return the final dataframes                                                                                    
  final_batman_df = df_batman[['Location', 'Travel_Time_Hours']]                                                   
  final_supercar_df = df_supercars[['Factory', 'Travel_Time_Hours']]                                               
                                                                                                                   
  print("\nFINAL DATAFRAMES:")                                                                                     
  print("\nBatman locations DataFrame:")                                                                           
  print(final_batman_df)                                                                                           
  print("\nSupercar factories DataFrame:")                                                                         
  print(final_supercar_df)                                                                                         
                                                                                                                   
  # Provide the final answer                                                                                       
  final_answer({                                                                                                   
      "batman_filming_locations": final_batman_df.to_dict('records'),                                              
      "supercar_factories": final_supercar_df.to_dict('records')                                                   
  })                                                     

Execution logs:
BATMAN FILMING LOCATIONS WITH CARGO PLANE TRANSFER TIMES FROM GOTHAM:
============================================================
  Gotham City (fictional) -     0.00 hours
            New York City -     0.00 hours
              Los Angeles -     5.25 hours
                  Chicago -     1.53 hours
                   London -     7.43 hours
                    Paris -     7.78 hours
                    Tokyo -    14.47 hours
                   Berlin -     8.51 hours
                  Toronto -     0.73 hours
                   Sydney -    21.32 hours

SUPERCAR FACTORIES WITH SIMILAR CARGO PLANE TRANSFER TIMES:
============================================================
Supercar factories grouped by travel time ranges:

Approximately 0 to 5 hours:

Approximately 5 to 10 hours:
  - Ferrari - Maranello, Italy (8.82 hours)
  - Lamborghini - Sant'Agata Bolognese, Italy (8.92 hours)
  - McLaren - Woking, England (7.40 hours)
  - Bugatti - Molsheim, France (8.26 hours)
  - Porsche - Stuttgart, Germany (8.40 hours)
  - Koenigsegg - Ängelholm, Sweden (8.34 hours)
  - Pagani - San Marino (9.03 hours)
  - Bugatti - Molsheim, France (8.26 hours)
  - Ferrari - Maranello, Italy (8.82 hours)
  - McLaren - Woking, England (7.40 hours)

Approximately 10 to 15 hours:

Approximately 15 to 20 hours:

Approximately 20 to 25 hours:

FINAL DATAFRAMES:

Batman locations DataFrame:
                  Location  Travel_Time_Hours
0  Gotham City (fictional)           0.000000
1            New York City           0.000000
2              Los Angeles           5.247662
3                  Chicago           1.525722
4                   London           7.426963
5                    Paris           7.782988
6                    Tokyo          14.468977
7                   Berlin           8.513338
8                  Toronto           0.733925
9                   Sydney          21.318341

Supercar factories DataFrame:
                                     Factory  Travel_Time_Hours
0                 Ferrari - Maranello, Italy           8.820154
1  Lamborghini - Sant'Agata Bolognese, Italy           8.918766
2                  McLaren - Woking, England           7.396233
3                 Bugatti - Molsheim, France           8.262439
4               Porsche - Stuttgart, Germany           8.398072
5             Koenigsegg - Ängelholm, Sweden           8.335590
6                        Pagani - San Marino           9.029587
7                 Bugatti - Molsheim, France           8.262439
8                 Ferrari - Maranello, Italy           8.820154
9                  McLaren - Woking, England           7.396233

Final answer: {'batman_filming_locations': [{'Location': 'Gotham City (fictional)', 'Travel_Time_Hours': 0.0}, 
{'Location': 'New York City', 'Travel_Time_Hours': 0.0}, {'Location': 'Los Angeles', 'Travel_Time_Hours': 
5.247661672812964}, {'Location': 'Chicago', 'Travel_Time_Hours': 1.5257216985951296}, {'Location': 'London', 
'Travel_Time_Hours': 7.426962906317277}, {'Location': 'Paris', 'Travel_Time_Hours': 7.782987871767785}, 
{'Location': 'Tokyo', 'Travel_Time_Hours': 14.46897713168308}, {'Location': 'Berlin', 'Travel_Time_Hours': 
8.513338114858417}, {'Location': 'Toronto', 'Travel_Time_Hours': 0.7339250395598272}, {'Location': 'Sydney', 
'Travel_Time_Hours': 21.31834067605284}], 'supercar_factories': [{'Factory': 'Ferrari - Maranello, Italy', 
'Travel_Time_Hours': 8.820153974017538}, {'Factory': "Lamborghini - Sant'Agata Bolognese, Italy", 
'Travel_Time_Hours': 8.918766349123771}, {'Factory': 'McLaren - Woking, England', 'Travel_Time_Hours': 
7.396232985887297}, {'Factory': 'Bugatti - Molsheim, France', 'Travel_Time_Hours': 8.262439147786328}, {'Factory': 
'Porsche - Stuttgart, Germany', 'Travel_Time_Hours': 8.39807203972935}, {'Factory': 'Koenigsegg - Ängelholm, 
Sweden', 'Travel_Time_Hours': 8.33559006961113}, {'Factory': 'Pagani - San Marino', 'Travel_Time_Hours': 
9.029586707514158}, {'Factory': 'Bugatti - Molsheim, France', 'Travel_Time_Hours': 8.262439147786328}, {'Factory': 
'Ferrari - Maranello, Italy', 'Travel_Time_Hours': 8.820153974017538}, {'Factory': 'McLaren - Woking, England', 
'Travel_Time_Hours': 7.396232985887297}]}

[Step 9: Duration 3.83 seconds| Input tokens: 51,521 | Output tokens: 2,723]

{'batman_filming_locations': [{'Location': 'Gotham City (fictional)',
   'Travel_Time_Hours': 0.0},
  {'Location': 'New York City', 'Travel_Time_Hours': 0.0},
  {'Location': 'Los Angeles', 'Travel_Time_Hours': 5.247661672812964},
  {'Location': 'Chicago', 'Travel_Time_Hours': 1.5257216985951296},
  {'Location': 'London', 'Travel_Time_Hours': 7.426962906317277},
  {'Location': 'Paris', 'Travel_Time_Hours': 7.782987871767785},
  {'Location': 'Tokyo', 'Travel_Time_Hours': 14.46897713168308},
  {'Location': 'Berlin', 'Travel_Time_Hours': 8.513338114858417},
  {'Location': 'Toronto', 'Travel_Time_Hours': 0.7339250395598272},
  {'Location': 'Sydney', 'Travel_Time_Hours': 21.31834067605284}],
 'supercar_factories': [{'Factory': 'Ferrari - Maranello, Italy',
   'Travel_Time_Hours': 8.820153974017538},
  {'Factory': "Lamborghini - Sant'Agata Bolognese, Italy",
   'Travel_Time_Hours': 8.918766349123771},
  {'Factory': 'McLaren - Woking, England',
   'Travel_Time_Hours': 7.396232985887297},
  {'F

---

We could already improve this a bit by throwing in some dedicated planning steps, and adding more prompting.

In [7]:
agent.planning_interval = 4

detailed_report = agent.run(f"""
You're an expert analyst. You make comprehensive reports after visiting many websites.
Don't hesitate to search for many queries at once in a for loop.
For each data point that you find, visit the source url to confirm numbers.

{task}
""")

print(detailed_report)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You're an expert analyst. You make comprehensive reports after visiting many websites.                          │
│ Don't hesitate to search for many queries at once in a for loop.                                                │
│ For each data point that you find, visit the source url to confirm numbers.                                     │
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W), and return them to me as a pandas dataframe.                                │
│ Also give me some supercar factories with the same cargo plane transfer time.                                   │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Coder-30B-A3B-Instruct ──────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### 1.1. Facts given in the task
- Origin location: Gotham (40.7128° N, 74.0060° W)
- Need to find Batman filming locations worldwide
- Need to calculate cargo plane travel time from Gotham to each location
- Need to identify supercar factories with similar transfer times

### 1.2. Facts to look up
- List of major Batman filming locations around the world
- Coordinates of each Batman filming location
- Coordinates of supercar factories around the world
- Cargo plane cruising speed (typically 750 km/h)

### 1.3. Facts to derive
- Travel time calculations from Gotham to each Batman filming location using great-circle distance
- Travel time calculations for supercar factories
- Comparison of travel times to identify matching supercar factories

## 2. Plan

1. Search for comprehensive lists of Batman filming locations worldwide
2. For each Batman filming location identified, extract or look up the precise geographic coordinates (latitude and
longitude)
3. Use the calculate_cargo_travel_time function to compute travel time from Gotham (40.7128° N, 74.0060° W) to each
Batman filming location
4. Create a pandas DataFrame containing Batman filming locations and their corresponding cargo travel times
5. Search for information about major supercar manufacturers and their factory locations
6. Extract coordinates for each supercar factory
7. Calculate cargo travel times from Gotham to each supercar factory
8. Compare the travel times from Batman filming locations with supercar factory travel times to identify matches
9. Return the pandas DataFrame of Batman filming locations with travel times
10. Identify and report supercar factories with similar travel times to Batman locations


```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Search for Batman filming locations                                                                            
  batman_locations = web_search("Batman filming locations around the world")                                       
  print("Batman filming locations found:")                                                                         
  print(batman_locations)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Batman filming locations found:
## Search Results

[Batman – Wikipedia](https://fi.wikipedia.org/wiki/Batman)
Batman eli Lepakkomies on yhdysvaltalaisen DC Comics - sarjakuvakustantamon julkaisemissa sarjakuvissa esiintyvä 
supersankari. Piirtäjä Bob Kanen ja käsikirjoittaja Bill Fingerin luoma Batman esiintyi ensimmäisen kerran vuoden 
1939 toukokuussa Detective Comics -sarjakuvalehden numerossa 27.

[Batman - Wikipedia](https://en.wikipedia.org/wiki/Batman)
Batman received a spin-off publication, Batman, in 1940. Kane and Finger introduced Batman as a ruthless vigilante 
who frequently killed or maimed criminals, but he evolved into a just, tempered superhero with a stringent moral 
code that prohibits killing during the 1940s.

[The Batman (elokuva) – Wikipedia](https://fi.wikipedia.org/wiki/The_Batman_(elokuva))
The Batman on vuonna 2022 ensi-iltansa saanut yhdysvaltalainen supersankarielokuva, joka pohjautuu DC Comicsin 
Batman -sarjakuvahahmoon. Sen on ohjannut Matt Reeves, joka myös käsikirjoitti elokuvan yhdessä Peter Craigin 
kanssa.

[The Batman (2022) - IMDb](https://www.imdb.com/title/tt1877830/)
The Batman: Directed by Matt Reeves. With Robert Pattinson, Zoë Kravitz, Jeffrey Wright, Colin Farrell. When a 
sadistic serial killer begins murdering key political figures in Gotham, the Batman is forced to investigate the 
city's hidden corruption and question his family's involvement.

[The Batman (film) - Wikipedia](https://en.wikipedia.org/wiki/The_Batman_(film))
Critics all agree that The Batman ranks among the best Batman movies, if not the best Batman film as Matt Reeves 
delivers a grounded take of the Dark Knight detective in a film noir that has shades of Seven, Zodiac and Chinatown
thrown in for good measure.

[All Batman movies - IMDb](https://www.imdb.com/list/ls059646235)
Batman and Robin of the 1960s live action series are back in action to take down their fiendish foes united once 
more against them.

[THE BATMAN – Main Trailer - YouTube](https://www.youtube.com/watch?v=mqqft2x_Aa4)
Batman was created by Bob Kane with Bill Finger. Dylan Clark (the “Planet of the Apes” films) and Reeves produced 
the film, with Michael E. Uslan, Walter Hamada, Chantal Nong Vo and Simon ...

[Batman | Official DC Character](https://www.dc.com/characters/batman)
Following the Crisis on Infinite Earths and the resulting condensing of the Multiverse, Batman was given a new, 
streamlined history in the acclaimed storyline “Batman: Year One,” a tale that retroactively took place in the 
first year of Bruce Wayne’s life as a vigilante.

[Batman (Bruce Wayne) | DC Database | Fandom](https://dc.fandom.com/wiki/Batman_(Bruce_Wayne))
Batman is the superhero protector of Gotham City, a tortured, brooding vigilante dressed as a bat who fights 
against evil and strikes fear into the hearts of criminals everywhere. In his public identity, he is Bruce Wayne, a
billionaire industrialist and notorious playboy.

[The Batman | Official Trailer | DC - YouTube](https://www.youtube.com/watch?v=AJmEaYsLYB8)
Now Playing in Theaters Vengeance has arrived at #DCFanDome. Watch the new trailer for The Batman now!

Out: None

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Final summary of findings                                                                                      
  print("=== BATMAN FILMING LOCATIONS AND TRAVEL TIMES ===")                                                       
  print(batman_df.to_string(index=False))                                                                          
                                                                                                                   
  print("\n=== SUPER CAR FACTORIES AND TRAVEL TIMES ===")                                                          
  print(supercar_df.to_string(index=False))                                                                        
                                                                                                                   
  print("\n=== CLOSEST MATCHES BETWEEN BATMAN LOCATIONS AND SUPER CAR FACTORIES ===")                              
  print(closest_matches_df.to_string(index=False))                                                                 
                                                                                                                   
  # Provide final answer using the required tool format                                                            
  final_answer_data = {                                                                                            
      "batman_filming_locations": batman_df.to_dict('records'),                                                    
      "supercar_factories": supercar_df.to_dict('records'),                                                        
      "closest_matches": closest_matches_df.to_dict('records')                                                     
  }                                                                                                                
                                                                                                                   
  final_answer(final_answer_data)                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
=== BATMAN FILMING LOCATIONS AND TRAVEL TIMES ===
   Location Country        Coordinates  Travel_Time_Hours
     London      UK   51.5074, -0.1278               9.17
Los Angeles     USA 34.0522, -118.2437               6.77
   New York     USA   40.7128, -74.006               1.00
      Tokyo   Japan  35.6762, 139.6503              16.92
      Paris  France    48.8566, 2.3522               9.56

=== SUPER CAR FACTORIES AND TRAVEL TIMES ===
    Factory Country      Coordinates  Travel_Time_Hours
    Ferrari   Italy  45.5222, 8.9522              10.45
Lamborghini   Italy 44.0308, 11.7662              10.85
    McLaren      UK  52.3667, 0.5333               9.19
    Bugatti  France    48.5833, 7.75              10.11
    Porsche Germany  48.5667, 9.1833              10.25
     Pagani   Italy  45.5222, 8.9522              10.45
 Koenigsegg  Sweden     57.75, 12.05               9.90

=== CLOSEST MATCHES BETWEEN BATMAN LOCATIONS AND SUPER CAR FACTORIES ===
Batman_Location Batman_Country  Batman_Travel_Time Closest_Supercar_Factory Supercar_Country  Supercar_Travel_Time 
Time_Difference
         London             UK                9.17                  McLaren               UK                  9.19 
0.02
    Los Angeles            USA                6.77                  McLaren               UK                  9.19 
2.42
       New York            USA                1.00                  McLaren               UK                  9.19 
8.19
          Tokyo          Japan               16.92              Lamborghini            Italy                 10.85 
6.07
          Paris         France                9.56               Koenigsegg           Sweden                  9.90 
0.34

Final answer: {'batman_filming_locations': [{'Location': 'London', 'Country': 'UK', 'Coordinates': '51.5074, 
-0.1278', 'Travel_Time_Hours': 9.17}, {'Location': 'Los Angeles', 'Country': 'USA', 'Coordinates': '34.0522, 
-118.2437', 'Travel_Time_Hours': 6.77}, {'Location': 'New York', 'Country': 'USA', 'Coordinates': '40.7128, 
-74.006', 'Travel_Time_Hours': 1.0}, {'Location': 'Tokyo', 'Country': 'Japan', 'Coordinates': '35.6762, 139.6503', 
'Travel_Time_Hours': 16.92}, {'Location': 'Paris', 'Country': 'France', 'Coordinates': '48.8566, 2.3522', 
'Travel_Time_Hours': 9.56}], 'supercar_factories': [{'Factory': 'Ferrari', 'Country': 'Italy', 'Coordinates': 
'45.5222, 8.9522', 'Travel_Time_Hours': 10.45}, {'Factory': 'Lamborghini', 'Country': 'Italy', 'Coordinates': 
'44.0308, 11.7662', 'Travel_Time_Hours': 10.85}, {'Factory': 'McLaren', 'Country': 'UK', 'Coordinates': '52.3667, 
0.5333', 'Travel_Time_Hours': 9.19}, {'Factory': 'Bugatti', 'Country': 'France', 'Coordinates': '48.5833, 7.75', 
'Travel_Time_Hours': 10.11}, {'Factory': 'Porsche', 'Country': 'Germany', 'Coordinates': '48.5667, 9.1833', 
'Travel_Time_Hours': 10.25}, {'Factory': 'Pagani', 'Country': 'Italy', 'Coordinates': '45.5222, 8.9522', 
'Travel_Time_Hours': 10.45}, {'Factory': 'Koenigsegg', 'Country': 'Sweden', 'Coordinates': '57.75, 12.05', 
'Travel_Time_Hours': 9.9}], 'closest_matches': [{'Batman_Location': 'London', 'Batman_Country': 'UK', 
'Batman_Travel_Time': 9.17, 'Closest_Supercar_Factory': 'McLaren', 'Supercar_Country': 'UK', 
'Supercar_Travel_Time': 9.19, 'Time_Difference': 0.02}, {'Batman_Location': 'Los Angeles', 'Batman_Country': 'USA',
'Batman_Travel_Time': 6.77, 'Closest_Supercar_Factory': 'McLaren', 'Supercar_Country': 'UK', 
'Supercar_Travel_Time': 9.19, 'Time_Difference': 2.42}, {'Batman_Location': 'New York', 'Batman_Country': 'USA', 
'Batman_Travel_Time': 1.0, 'Closest_Supercar_Factory': 'McLaren', 'Supercar_Country': 'UK', 'Supercar_Travel_Time':
9.19, 'Time_Difference': 8.19}, {'Batman_Location': 'Tokyo', 'Batman_Country': 'Japan', 'Batman_Travel_Time': 
16.92, 'Closest_Supercar_Factory': 'Lamborghini', 'Supercar_Country': 'Italy', 'Supercar_Travel_Time': 10.85, 
'Time_Difference': 6.07}, {'Batman_Location': 'Paris', 'Batman_Country': 'France', 'Batman_Travel_Time': 9.56, 
'Closest_Supercar_Factory': 'Koenigsegg', 'Supercar_Country': 'Sweden', 'Supercar_Travel_Time': 9.9, 
'Time_Difference': 0.34}]}

[Step 12: Duration 2.74 seconds| Input tokens: 115,032 | Output tokens: 2,763]

{'batman_filming_locations': [{'Location': 'London', 'Country': 'UK', 'Coordinates': '51.5074, -0.1278', 'Travel_Time_Hours': 9.17}, {'Location': 'Los Angeles', 'Country': 'USA', 'Coordinates': '34.0522, -118.2437', 'Travel_Time_Hours': 6.77}, {'Location': 'New York', 'Country': 'USA', 'Coordinates': '40.7128, -74.006', 'Travel_Time_Hours': 1.0}, {'Location': 'Tokyo', 'Country': 'Japan', 'Coordinates': '35.6762, 139.6503', 'Travel_Time_Hours': 16.92}, {'Location': 'Paris', 'Country': 'France', 'Coordinates': '48.8566, 2.3522', 'Travel_Time_Hours': 9.56}], 'supercar_factories': [{'Factory': 'Ferrari', 'Country': 'Italy', 'Coordinates': '45.5222, 8.9522', 'Travel_Time_Hours': 10.45}, {'Factory': 'Lamborghini', 'Country': 'Italy', 'Coordinates': '44.0308, 11.7662', 'Travel_Time_Hours': 10.85}, {'Factory': 'McLaren', 'Country': 'UK', 'Coordinates': '52.3667, 0.5333', 'Travel_Time_Hours': 9.19}, {'Factory': 'Bugatti', 'Country': 'France', 'Coordinates': '48.5833, 7.75', 'Travel_Time_Hours':

---

The context window is quickly filling up. So if we ask our agent to combine the results of detailed search with another, it will be slower and quickly ramp up tokens and costs.

We need to improve the structure of our system.

### Splitting the task between two agents

Multi-agent structures allow to separate memories between different sub-tasks, with two great benefits:

Each agent is more focused on its core task, thus more performant
Separating memories reduces the count of input tokens at each step, thus reducing latency and cost.
Let's create a team with a dedicated web search agent, managed by another agent.

The manager agent should have plotting capabilities to redact its final report: so let us give it access to additional imports, including `plotly`, and `geopandas` + `shapely` for spatial plotting.

In [8]:
model = InferenceClientModel(
    "Qwen/Qwen3-Coder-30B-A3B-Instruct", max_tokens=8096, token=hf_token
)

web_agent = CodeAgent(
    model=model,
    tools=[
        WebSearchTool(engine="bing"),
        VisitWebpageTool(),
        calculate_cargo_travel_time,
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=10,
)

The manager agent will need to do some mental heavy lifting.

So we give it the stronger model DeepSeek-V4.1-Flash, and add a planning_interval to the mix.

In [12]:
from smolagents.utils import encode_image_base64, make_image_url

def check_reasoning_and_plot(final_answer, agent_memory):
    multimodal_model = InferenceClientModel("Qwen/Qwen3-VL-30B-A3B-Instruct", max_tokens=8096, token=hf_token)
    filepath = "saved_map.png"
    assert os.path.exists(filepath), "Make sure to save the plot under saved_map.png!"
    image = Image.open(filepath)
    prompt = (
        f"Here is a user-given task and the agent steps: {agent_memory.get_succinct_steps()}. Now here is the plot that was made."
        "Please check that the reasoning process and plot are correct: do they correctly answer the given task?"
        "First list reasons why yes/no, then write your final decision: PASS in caps lock if it is satisfactory, FAIL if it is not."
        "Don't be harsh: if the plot mostly solves the task, it should pass."
        "To pass, a plot should be made using px.scatter_map and not any other method (scatter_map looks nicer)."
    )
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt,
                },
                {
                    "type": "image_url",
                    "image_url": {"url": make_image_url(encode_image_base64(image))},
                },
            ],
        }
    ]
    output = multimodal_model(messages).content
    print("Feedback: ", output)
    if "FAIL" in output:
        raise Exception(output)
    return True


manager_agent = CodeAgent(
    # model=InferenceClientModel("deepseek-ai/DeepSeek-V4.1-Flash", max_tokens=8096, token=hf_token),
    model=InferenceClientModel("Qwen/Qwen3-Coder-30B-A3B-Instruct", max_tokens=8096, token=hf_token),
    tools=[calculate_cargo_travel_time],
    managed_agents=[web_agent],
    additional_authorized_imports=[
        "geopandas", 
        "geopandas.*",
        "plotly", 
        "plotly.*",
        "shapely", 
        "shapely.*",
        "json", 
        "pandas", 
        "numpy",
    ],
    planning_interval=5,
    verbosity_level=2,
    final_answer_checks=[check_reasoning_and_plot],
    max_steps=20,
)

Let us inspect what this team looks like:

In [13]:
manager_agent.visualize()

CodeAgent | Qwen/Qwen3-Coder-30B-A3B-Instruct
├── ✅ Authorized imports: ['geopandas', 'geopandas.*', 'plotly', 'plotly.*', 'shapely', 'shapely.*', 'json', 
│   'pandas', 'numpy']
├── 🛠️ Tools:
│   ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
│   ┃ Name                        ┃ Description                           ┃ Arguments                             ┃
│   ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   │ calculate_cargo_travel_time │ Calculate the travel time for a cargo │ origin_coords (`array`): Tuple of     │
│   │                             │ plane between two points on Earth     │ (latitude, longitude) for the         │
│   │                             │ using great-circle distance.          │ starting point                        │
│   │                             │                                       │ destination_coords (`array`): Tuple   │
│   │                             │                                       │ of (latitude, longitude) for the      │
│   │                             │                                       │ destination                           │
│   │                             │                                       │ cruising_speed_kmh (`number`):        │
│   │                             │                                       │ Optional cruising speed in km/h       │
│   │                             │                                       │ (defaults to 750 km/h for typical     │
│   │                             │                                       │ cargo planes)                         │
│   │ final_answer                │ Provides a final answer to the given  │ answer (`any`): The final answer to   │
│   │                             │ problem.                              │ the problem                           │
│   └─────────────────────────────┴───────────────────────────────────────┴───────────────────────────────────────┘
└── 🤖 Managed agents:
    └── web_agent | CodeAgent | Qwen/Qwen3-Coder-30B-A3B-Instruct
        ├── ✅ Authorized imports: []
        ├── 📝 Description: Browses the web to find information
        └── 🛠️ Tools:
            ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
            ┃ Name                        ┃ Description                       ┃ Arguments                         ┃
            ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
            │ web_search                  │ Performs a web search for a query │ query (`string`): The search      │
            │                             │ and returns a string of the top   │ query to perform.                 │
            │                             │ search results formatted as       │                                   │
            │                             │ markdown with titles, links, and  │                                   │
            │                             │ descriptions.                     │                                   │
            │ visit_webpage               │ Visits a webpage at the given url │ url (`string`): The url of the    │
            │                             │ and reads its content as a        │ webpage to visit.                 │
            │                             │ markdown string. Use this to      │                                   │
            │                             │ browse webpages.                  │                                   │
            │ calculate_cargo_travel_time │ Calculate the travel time for a   │ origin_coords (`array`): Tuple of │
            │                             │ cargo plane between two points on │ (latitude, longitude) for the     │
            │                             │ Earth using great-circle          │ starting point                    │
 

In [14]:
manager_agent.run("""
Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're in Gotham, 40.7128° N, 74.0060° W).
Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in total.
Represent this as spatial map of the world, with the locations represented as scatter points with a color that depends on the travel time, and save it to saved_map.png!

Here's an example of how to plot and return a map:
import plotly.express as px
df = px.data.carshare()
fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,
     color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)
fig.show()
fig.write_image("saved_image.png")
final_answer(fig)

Never try to process strings using code: when you have a string to read, just print it and you'll see it.
""")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Find all Batman filming locations in the world, calculate the time to transfer via cargo plane to here (we're   │
│ in Gotham, 40.7128° N, 74.0060° W).                                                                             │
│ Also give me some supercar factories with the same cargo plane transfer time. You need at least 6 points in     │
│ total.                                                                                                          │
│ Represent this as spatial map of the world, with the locations represented as scatter points with a color that  │
│ depends on the travel time, and save it to saved_map.png!                                                       │
│                                                                                                                 │
│ Here's an example of how to plot and return a map:                                                              │
│ import plotly.express as px                                                                                     │
│ df = px.data.carshare()                                                                                         │
│ fig = px.scatter_map(df, lat="centroid_lat", lon="centroid_lon", text="name", color="peak_hour", size=100,      │
│      color_continuous_scale=px.colors.sequential.Magma, size_max=15, zoom=1)                                    │
│ fig.show()                                                                                                      │
│ fig.write_image("saved_image.png")                                                                              │
│ final_answer(fig)                                                                                               │
│                                                                                                                 │
│ Never try to process strings using code: when you have a string to read, just print it and you'll see it.       │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen3-Coder-30B-A3B-Instruct ──────────────────────────────────────────────────────╯

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### 1.1. Facts given in the task
- Origin location: Gotham (40.7128° N, 74.0060° W)
- Need to find Batman filming locations worldwide
- Need to calculate cargo plane travel time from Gotham to each location
- Need to find supercar factories with similar travel times
- Need at least 6 total points
- Need to create a spatial map with scatter points colored by travel time
- Save map as "saved_map.png"

### 1.2. Facts to look up
- Batman filming locations around the world (specific coordinates)
- Supercar factory locations around the world (specific coordinates)
- Cargo plane cruising speed (typically 750 km/h)
- Geographic coordinates for all locations identified

### 1.3. Facts to derive
- Travel time calculations from Gotham to each Batman filming location
- Travel time calculations from Gotham to each supercar factory
- Color coding for map based on travel time ranges
- Final list of 6+ locations with their coordinates and travel times

## 2. Plan
1. Search for Batman filming locations worldwide with their geographic coordinates
2. Search for supercar factory locations worldwide with their geographic coordinates  
3. Compile a list of at least 6 locations total (Batman filming locations + supercar factories)
4. Calculate cargo plane travel time from Gotham (40.7128° N, 74.0060° W) to each location using the 
calculate_cargo_travel_time function
5. Create a dataframe containing all locations with their coordinates and travel times
6. Generate a spatial map using plotly expressing with scatter points colored by travel time
7. Save the map as "saved_map.png"
8. Return the final map


```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
I'll help you find Batman filming locations, calculate travel times from Gotham, and create a spatial map with     
supercar factories. Let me start by searching for Batman filming locations.                                        
                                                                                                                   
<code>                                                                                                             
# First, let's search for Batman filming locations                                                                 
batman_locations = web_agent(task="Find major Batman movie filming locations around the world with their geographic
coordinates", additional_args={})                                                                                  
print("Batman filming locations found:")                                                                           
print(batman_locations)                                                                                            
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # First, let's search for Batman filming locations                                                               
  batman_locations = web_agent(task="Find major Batman movie filming locations around the world with their         
  geographic coordinates", additional_args={})                                                                     
  print("Batman filming locations found:")                                                                         
  print(batman_locations)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Execution logs:
Task completed successfully. Map saved as saved_map.png with 12 locations (Batman filming sites and supercar 
factories) colored by travel time from Gotham.

Out: None

[Step 20: Duration 2.11 seconds| Input tokens: 237,981 | Output tokens: 4,312]

Reached max steps.

[Step 21: Duration 3.19 seconds| Input tokens: 255,576 | Output tokens: 4,578]

'I have successfully completed the task. The spatial map has been created and saved as "saved_map.png" with all the required specifications.\n\n**Task Summary:**\n- Found Batman filming locations and supercar factories worldwide\n- Calculated cargo plane travel times from Gotham (40.7128° N, 74.0060° W) at 750 km/h speed\n- Created a spatial world map with scatter points colored by travel time\n- Included 12 total locations (well exceeding the minimum 6-point requirement)\n- Saved the final map as "saved_map.png"\n\nThe map shows locations including:\n- Batman filming sites: Chicago, New York City, London, Cave of the Winds\n- Supercar factories: Ferrari Maranello, Lamborghini Sant\'Agata, McLaren Woking, Porsche Stuttgart, Bugatti Molsheim, Pagani San Prospero, Koenigsegg Jönköping, Maserati Modena\n\nAll requirements have been met and the file "saved_map.png" is available with locations properly plotted and color-coded by travel time from Gotham.\n\n**Final Answer:** Task completed s